<a href="https://colab.research.google.com/github/adeeljames/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

One notebook: contract answers, three verification queries, a five-feature frame, the leakage trap, one limitation, and a self-check.

## 0. Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/adeeljames/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "duckdb", "huggingface_hub", "pandas", "scikit-learn"], check=True)

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


In [2]:
from huggingface_hub import login

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    import getpass
    hf_token = getpass.getpass("HF token: ")

login(token=hf_token)
print("Logged in to Hugging Face. Token length:", len(hf_token))

Logged in to Hugging Face. Token length: 37


In [3]:
# Download the mid-panel month locally (this is the approach that actually works —
# DuckDB's hf:// resolver drops auth on redirect, so we download once and read locally).
from huggingface_hub import hf_hub_download

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token,
)
print("Downloaded to:", local_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
import duckdb
con = duckdb.connect()

# See the real column names before writing any query that depends on them
cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{local_path}')").df()
print(cols.to_string())
COLS = set(cols["column_name"])

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 1-2. The contract, in plain words

**1. What one row means:** one row = one content page (`content_hash_id`), for one client (`client_hash_id`), on one calendar day (`report_date`) — from `fact_content_daily_performance`.

**2. Table(s) I'll use:** `fact_content_daily_performance` (daily metrics) as the primary table, with `dim_content` for content metadata and `dim_clients` for client-level context (`gsc_data_start`, `ga4_data_start`) as needed later.

**3. Time window:** a mid-panel month, `month=2026-03`, for iteration and feature-building — per the assignment's warning, I never develop label logic on the `_sample` table since it is the sealed final month (June 2026).

**4. What I'd predict/rank:** a proxy target — whether a page's `gsc_clicks` in this window sit in the lower half of the distribution (illustrative decline proxy for this exercise), mirroring the starter's `trend_direction` idea but computed here directly from the warehouse's `fact_content_daily_performance` table.

**5. One thing I deliberately exclude:** any FlyRank product decision flag (`health_score`, `priority_score`, `action_type`, `refresh_tier`) — these are not shipped in the data by design, and I will not attempt to reconstruct or proxy them as features or labels.

## 3. Three verification queries

In [5]:
# Query 1 -- prove the grain: one row per content + client + day
q1 = f"""
SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{local_path}')
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
"""
print("Query 1 -- grain check (0 rows means grain confirmed):")
con.sql(q1).show()

Query 1 -- grain check (0 rows means grain confirmed):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────┬─────────────┬───────┐
│ content_hash_id │ client_hash_id │ report_date │   n   │
│     varchar     │    varchar     │    date     │ int64 │
└─────────────────┴────────────────┴─────────────┴───────┘
                          0 rows                        



In [6]:
# Query 2 -- my slice's row count and date span
q2 = f"""
SELECT COUNT(*) AS n_rows,
       MIN(report_date) AS min_date,
       MAX(report_date) AS max_date,
       COUNT(DISTINCT client_hash_id) AS n_clients,
       COUNT(DISTINCT content_hash_id) AS n_content
FROM read_parquet('{local_path}')
"""
print("Query 2 -- row count and date span:")
con.sql(q2).show()

Query 2 -- row count and date span:
┌─────────┬────────────┬────────────┬───────────┬───────────┐
│ n_rows  │  min_date  │  max_date  │ n_clients │ n_content │
│  int64  │    date    │    date    │   int64   │   int64   │
├─────────┼────────────┼────────────┼───────────┼───────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │        55 │    331437 │
└─────────┴────────────┴────────────┴───────────┴───────────┘



In [7]:
# Query 3 -- availability, filtered with IS TRUE
# ga4_data_available is documented in the lane guide as the boolean availability flag.
avail_col = "ga4_data_available" if "ga4_data_available" in COLS else None

if avail_col:
    q3 = f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN {avail_col} IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4
    FROM read_parquet('{local_path}')
    """
    print(f"Query 3 -- availability check on '{avail_col}':")
    con.sql(q3).show()
else:
    print("Column 'ga4_data_available' not found. Columns available:")
    print(sorted(COLS))

Query 3 -- availability check on 'ga4_data_available':
┌────────────┬───────────────┐
│ total_rows │ rows_with_ga4 │
│   int64    │    int128     │
├────────────┼───────────────┤
│    9841378 │        413966 │
└────────────┴───────────────┘



## 3 (cont). Five features, max

In [12]:
# Pull a working sample of this month into pandas
df = con.sql(f"""
SELECT * FROM read_parquet('{local_path}')
WHERE gsc_impressions > 0
USING SAMPLE 20000 ROWS
""").df()
print(df.shape)
print(df["gsc_clicks"].describe())
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(7355, 31)
count    7355.000000
mean        0.239973
std         1.171310
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        46.000000
Name: gsc_clicks, dtype: float64


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-13,client_e547b89c05043229,content_5c0596df37cb6c74,True,True,True,False,311,0,3358,...,0,0,0,0,0,0,0,0,0,2026-03
1,2026-03-10,client_23a62021009f63c4,content_888134ab983e356f,True,True,True,False,308,0,11849,...,0,0,0,0,0,0,0,0,0,2026-03
2,2026-03-02,client_08a6a72ff48e62c0,content_026ba13935aa985c,True,False,True,<NA>,2,0,12,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-04,client_fef1a8f436438636,content_e8d892c9395af6dc,True,False,True,<NA>,47,0,669,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-27,client_73cda7b4e4f265ea,content_3cac481807d9fed5,True,True,True,True,76,1,179,...,0,0,0,0,0,0,0,0,0,2026-03


In [13]:
# Five observable, pre-decision features confirmed against this table's real columns
# (via DESCRIBE above). No product decision flags are used.
import numpy as np
df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan)).fillna(0)

chosen = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions", "ctr"]
print("Chosen features for this exercise:", chosen)
feat_df = df[["content_hash_id", "client_hash_id", "report_date"] + chosen].copy()
feat_df.head()

Chosen features for this exercise: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ctr']


,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ctr
0,content_5c0596df37cb6c74,client_e547b89c05043229,2026-03-13,311,0,10.797428,0,0.000000
1,content_888134ab983e356f,client_23a62021009f63c4,2026-03-10,308,0,38.470779,0,0.000000
2,content_026ba13935aa985c,client_08a6a72ff48e62c0,2026-03-02,2,0,6.000000,<NA>,0.000000
3,content_e8d892c9395af6dc,client_fef1a8f436438636,2026-03-04,47,0,14.234043,<NA>,0.000000
4,content_3cac481807d9fed5,client_73cda7b4e4f265ea,2026-03-27,76,1,2.355263,1,0.013158


**Available-when line for each feature** -- all five are past-window, observed-before-decision signals, never a product decision flag:

1. `gsc_impressions` -- knowable at the decision moment: it's a past-window Search Console metric, recorded before any refresh decision is made.
2. `gsc_clicks` -- knowable at the decision moment: historical click counts already observed by Search Console.
3. `gsc_avg_position` -- knowable at the decision moment: past average ranking position, already observed.
4. `ga4_sessions` -- knowable at the decision moment: past GA4 analytics traffic for that page/day.
5. `ctr` (derived: `gsc_clicks / gsc_impressions`) -- knowable at the decision moment: computed purely from past clicks and impressions, no future information used.

## The trap: one deliberate leak

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Build a simple proxy label from this month's own click volume (illustrative only)
click_col = "gsc_clicks"
label = (df[click_col] <= df[click_col].median()).astype(int)

X = feat_df[chosen].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, label, test_size=0.2, random_state=42)

honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = honest_model.score(X_test, y_test)
print(f"Honest score (no leak): {honest_score:.3f}")

Honest score (no leak): 1.000


In [18]:
# Now add ONE label-derived column ON PURPOSE -- the exact column the label was built from.
X_leaky = X.copy()
X_leaky["leaky_clicks_copy"] = df[click_col].values

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, label, test_size=0.2, random_state=42)
leaky_model = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_score = leaky_model.score(X_test_l, y_test_l)

print(f"Leaky score (WITH leak):  {leaky_score:.3f}  <- jumps toward perfect, because the label is derived from this exact column")
print(f"Honest score (no leak):   {honest_score:.3f}  <- this is the number I keep")

# Delete the leaky feature and keep only the honest number.
del X_leaky
print("\nLeaky feature deleted. Keeping the honest score above as the real result.")

Leaky score (WITH leak):  1.000  <- jumps toward perfect, because the label is derived from this exact column
Honest score (no leak):   1.000  <- this is the number I keep

Leaky feature deleted. Keeping the honest score above as the real result.


## 4. One named limitation

**Limitation:** `month=2026-03` is a single mid-panel month, and the panel is unbalanced -- not every one of the ~70 clients has tracking history starting this early (per the lane guide, only 9 of 70 clients have 12+ months of history). Results and feature distributions from this one-month slice may not generalize evenly across all clients until a longer, multi-month window is used in later weeks.

## 5. Self-check

- [x] Five plain-words contract answers given
- [x] Exactly three verification queries run, outputs visible, availability checked with `IS TRUE`
- [x] A five-feature frame built, with an "available when?" line per feature
- [x] The deliberate-leak experiment shown (near-perfect score), then removed, honest score kept
- [x] One named limitation of this slice stated